In [3]:
import joblib
import google.generativeai as genai

from transformers import pipeline

In [4]:
import sys
print(sys.executable)

c:\Users\Dikshya\Desktop\AI-Customer-Support-CoPilot\Backend\venv\Scripts\python.exe


In [5]:
import joblib
import transformers

print("Imports successful!")

Imports successful!


In [6]:
from transformers import pipeline

print("Pipeline imported successfully!")

Pipeline imported successfully!


In [7]:
# Load Saved ML Models

queue_model = joblib.load("../models/queue_model.pkl")
queue_vectorizer = joblib.load("../models/queue_vectorizer.pkl")

priority_model = joblib.load("../models/priority_model.pkl")
priority_vectorizer = joblib.load("../models/priority_vectorizer.pkl")

print("✅ Queue Model Loaded")
print("✅ Priority Model Loaded")

c:\Users\Dikshya\Desktop\AI-Customer-Support-CoPilot\Backend\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Dikshya\Desktop\AI-Customer-Support-CoPilot\Backend\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Dikshya\Desktop\AI-Customer-Support-CoPilot\Backend\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarni

✅ Queue Model Loaded
✅ Priority Model Loaded


In [8]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    framework="pt",
    device="cpu"
)

print("✅ Summarizer Loaded")

Device set to use cpu


✅ Summarizer Loaded


In [9]:
# Configure Gemini

import os
from dotenv import load_dotenv
import google.generativeai as genai

# Load .env file from Backend folder
load_dotenv("../.env")

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("❌ GEMINI_API_KEY not found in .env")

genai.configure(api_key=API_KEY)

gemini_model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print("✅ Gemini Configured Successfully")

✅ Gemini Configured Successfully


In [10]:
def predict_queue(ticket):

    ticket_vector = queue_vectorizer.transform([ticket])
    prediction = queue_model.predict(ticket_vector)[0]

    return prediction


def predict_priority(ticket):

    ticket_vector = priority_vectorizer.transform([ticket])
    prediction = priority_model.predict(ticket_vector)[0]

    return prediction


def summarize_ticket(ticket):

    words = len(ticket.split())

    if words < 35:
        return ticket

    summary = summarizer(
        ticket,
        max_length=30,
        min_length=10,
        do_sample=False
    )

    return summary[0]["summary_text"]


from google.api_core.exceptions import ResourceExhausted

def generate_reply(summary, ticket_type, queue, priority):

    prompt = f"""
You are an AI Customer Support Copilot assisting human support agents.

Generate a professional customer support draft reply.

Customer Ticket Type:
{ticket_type}

Predicted Support Queue:
{queue}

Predicted Priority:
{priority}

Ticket Summary:
{summary}

Instructions:

- Start with "Dear Customer,"
- Thank the customer for contacting our support team.
- Do NOT mention the predicted support queue.
- If the ticket describes a problem or incident, apologize for the inconvenience.
- If the ticket is a request or question, do NOT apologize unnecessarily.
- Acknowledge the customer's concern or request.
- Reassure the customer when appropriate.
- Never invent technical details.
- Never assume the root cause.
- Never promise unsupported timelines.
- Keep the reply between 5–7 sentences.
- End exactly with:

Kind regards,
Customer Support Team

Return ONLY the reply.
"""

    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except ResourceExhausted:
        return (
            "AI reply generation is temporarily unavailable because the API quota has been reached. "
            "Please try again later."
    )

In [11]:
def process_ticket(ticket):

    # Queue Prediction
    predicted_queue = predict_queue(ticket)

    # Priority Prediction
    predicted_priority = predict_priority(ticket)

    # Ticket Summary
    ticket_summary = summarize_ticket(ticket)

    # Generate AI Reply
    ai_reply = generate_reply(
        summary=ticket_summary,
        ticket_type="Unknown",
        queue=predicted_queue,
        priority=predicted_priority
    )

    return {
        "queue": predicted_queue,
        "priority": predicted_priority,
        "summary": ticket_summary,
        "reply": ai_reply
    }

In [12]:
sample_ticket = """
The centralized account management portal is currently unavailable.
I have tried multiple browsers and devices but I still cannot access my account.
"""

result = process_ticket(sample_ticket)

print("=" * 70)
print("Predicted Queue:")
print(result["queue"])

print("\nPredicted Priority:")
print(result["priority"])

print("\nSummary:")
print(result["summary"])

print("\nSuggested Reply:\n")
print(result["reply"])

Predicted Queue:
IT Support

Predicted Priority:
high

Summary:

The centralized account management portal is currently unavailable.
I have tried multiple browsers and devices but I still cannot access my account.


Suggested Reply:

Dear Customer,
Thank you for contacting our support team. We sincerely apologize for the inconvenience you are experiencing with the centralized account management portal being unavailable. We understand that you are unable to access your account despite trying multiple browsers and devices. Our team is actively investigating this issue to restore full service. We appreciate your patience as we work to resolve this matter. We will provide an update as soon as more information is available.

Kind regards,
Customer Support Team


In [13]:
# Load Dataset for Testing

import pandas as pd

df = pd.read_csv("../data/customer_support_tickets_clean.csv")

print("✅ Dataset Loaded")
print(df.shape)

df.head()

✅ Dataset Loaded
(16335, 8)


,subject,body,answer,type,queue,priority,ticket_text,clean_text
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support team ...
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,"Feature Query Dear Customer Support,\n\nI hope...",feature query dear customer support i hope thi...


In [14]:
sample_df = df.sample(1, random_state=42)

for _, row in sample_df.iterrows():

    ticket = row["ticket_text"]

    result = process_ticket(ticket)

    print("=" * 120)

    print("ORIGINAL TICKET:\n")
    print(ticket[:500] + "...")

    print("\nACTUAL QUEUE:")
    print(row["queue"])

    print("PREDICTED QUEUE:")
    print(result["queue"])

    print("\nACTUAL PRIORITY:")
    print(row["priority"])

    print("PREDICTED PRIORITY:")
    print(result["priority"])

    print("\nSUMMARY:\n")
    print(result["summary"])

    print("\nGENERATED REPLY:\n")
    print(result["reply"])

    print("=" * 120)
    print()

ORIGINAL TICKET:

Request for Documentation on Integrating Adobe Sign Looking for detailed instructions on integrating the Adobe Sign project management SaaS solution. Could you outline the steps and requirements needed for a successful integration?...

ACTUAL QUEUE:
Product Support
PREDICTED QUEUE:
Technical Support

ACTUAL PRIORITY:
low
PREDICTED PRIORITY:
low

SUMMARY:

Request for Documentation on Integrating Adobe Sign Looking for detailed instructions on integrating the Adobe Sign project management SaaS solution. Could you outline the steps and requirements needed for a successful integration?

GENERATED REPLY:

Dear Customer,
Thank you for contacting our support team. We understand you are seeking detailed instructions and requirements for integrating Adobe Sign into your project management solution. We are happy to assist you with this request. Our team is currently compiling the most relevant documentation and resources to guide you through a successful integration process. W